# Hoot — EDA (Etapa 11a)

Podaci iz `uv run python -m hoot_ml.export` (`ml/data/`). Oznake bloka: odrađen / preskočen / nepotvrđen (RESEARCH-ML 1.2); nepotvrđen nije neuspjeh.

In [ ]:
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from hoot_ml.data import block_examples, load, session_minutes

tables = load(Path("../data"))
today = date.today().isoformat()
{name: len(df) for name, df in tables.items()}

## Blokovi: oznake po dijelu dana i opterećenju

In [ ]:
blocks = block_examples(tables, today)
blocks["label"].value_counts()

In [ ]:
pd.crosstab(blocks["part"], blocks["label"], normalize="index").round(2)

In [ ]:
blocks["load_bin"] = pd.cut(blocks["day_load_min"], [0, 60, 120, 180, 240, 600])
pd.crosstab(blocks["load_bin"], blocks["label"], normalize="index").round(2)

In [ ]:
pd.crosstab(blocks["weekday"], blocks["label"], normalize="index").round(2)

## Trajanje: stvarno vs planirano (planning fallacy)

In [ ]:
sessions = tables["study_sessions"].set_index("id")
sessions["active_min"] = session_minutes(tables)
planned = sessions.dropna(subset=["planned_minutes", "active_min"])
ratio = planned["active_min"] / planned["planned_minutes"]
ratio.describe()

In [ ]:
ax = ratio.clip(0, 3).plot.hist(bins=30)
ax.set_xlabel("aktivno / planirano")
plt.show()

## Ometanja

In [ ]:
sessions[["app_leaves", "app_away_sec", "distractions", "focus"]].describe()

## Ponavljanja (FSRS)

In [ ]:
logs = tables["review_logs"]
logs["rating"].value_counts().sort_index()

In [ ]:
seen = logs.dropna(subset=["retrievability_before"]).copy()
seen["r_bin"] = pd.cut(seen["retrievability_before"], [0, 0.5, 0.7, 0.8, 0.9, 1])
# Calibration: predicted R against the share not rated Again.
seen.groupby("r_bin", observed=True).agg(
    n=("rating", "size"), recalled=("rating", lambda r: (r > 1).mean())
)